# 🎉 Free Remote vLLM Demo on Google Colab

This notebook installs **vllm**, launches the **Meta‑Llama‑3.1‑8B‑Instruct** model, and exposes the OpenAI‑compatible API through a free **ngrok** tunnel.

You can then point your local `text‑to‑sql‑eval‑lab` `.env` to the ngrok URL and run evaluations against the remote model.

> **Note**: You need a HuggingFace token with access to the Llama model. Store it in the `HF_TOKEN` environment variable in the first cell.

In [ ]:
%pip install -q vllm pyngrok
import os, subprocess, threading, time
from pyngrok import ngrok

# ---- 1️⃣ Set your HuggingFace token ----
# Replace <YOUR_TOKEN> with your actual token or set it as an environment variable before running.
HF_TOKEN = os.getenv('HF_TOKEN') or '<YOUR_TOKEN>'
os.environ['HF_TOKEN'] = HF_TOKEN

# ---- 2️⃣ Function to run vllm server ----
def run_vllm():
    cmd = ["vllm", "serve", "meta-llama/Meta-Llama-3.1-8B-Instruct",
           "--host", "0.0.0.0", "--port", "8000",
           "--enable-prefix-caching"]
    # Use subprocess to keep the process alive in the background
    subprocess.Popen(cmd)

# Run vllm in a separate thread so the notebook stays responsive
threading.Thread(target=run_vllm, daemon=True).start()

# ---- 3️⃣ Give the server a moment to start ----
time.sleep(5)

# ---- 4️⃣ Open an ngrok tunnel ----
public_url = ngrok.connect(8000, bind_tls=True).public_url
print("🔗 ngrok tunnel URL (OpenAI API base):", public_url + "/v1")

# ---- 5️⃣ Test the endpoint (optional) ----
# import requests
# resp = requests.get(public_url + '/v1/models')
# print(resp.json())
